<a href="https://colab.research.google.com/github/Kireeti2808/agri_doctor/blob/main/Agri_Doctor_rice_v14.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
# 0. Settings
import os
SEEDS = None                         # None: 3 seeds on A100, 1 seed (42) otherwise
HASH_THRESHOLD = 4                   # the paper's duplicate rule
# no embedding / labelling step: only the paper's test photos and their exact 64-bit hash copies are held out
LAB_FRACTIONS = {'val': 0.15, 'test': 0.15}
FIELD_FRACTIONS = {'val': 0.15}
HASH_SKIP_SOURCES = {'canopy'}     # uniform canopy scenes collide under the 8x8 hash (audit fix 2.5): embeddings only there
V14 = dict(size=300, batch=32, epochs_head=5, epochs_full=30, lr_head=1e-3, lr_full=2e-4, patience=6, smoothing=0.1,
           backbone='EfficientNetV2S', weights='imagenet')
EXTRA_DIR_NAME = 'agri_extra_rice'      # put the Mendeley zips here (optional; the run works without them)
EXTRA_TEST_FRACTION = 0.2               # held-out share of each new dataset (split by hash group)
CROSS_SOURCE_HOLDOUT = 'riceleafbd'     # also train one model without this dataset and test on it (None to skip)

In [2]:
# 1. Environment: GPU, Drive, repository, packages
!nvidia-smi -L || echo "NO GPU: training will be very slow. Runtime > Change runtime type > GPU"
import sys, json, time, shutil, subprocess, zipfile, random, re, hashlib, platform
from pathlib import Path
from collections import Counter, defaultdict
import numpy as np
try:
    from google.colab import drive, userdata
    drive.mount('/content/drive'); BASE = Path('/content/drive/MyDrive')
except Exception as e:
    print('Google Drive not available, saving locally:', e); BASE = Path('/content/local_drive'); BASE.mkdir(parents=True, exist_ok=True)
OUT = BASE / 'agri_v14'; MODELS = OUT / 'models'; MODELS.mkdir(parents=True, exist_ok=True)
for k in ('KAGGLE_USERNAME', 'KAGGLE_KEY'):
    try:
        v = userdata.get(k)
        if v: os.environ[k] = v
    except Exception:
        pass
if not Path('/content/agri_doctor').exists():
    !git clone -q https://github.com/Kireeti2808/agri_doctor /content/agri_doctor
!pip install -q kagglehub gdown transformers scipy scikit-learn
sys.path[:0] = ['/content', '/content/agri_doctor', '/content/agri_doctor/training']
import tensorflow as tf
GPU = tf.config.list_physical_devices('GPU')
try:
    gpu_name = subprocess.run(['nvidia-smi', '--query-gpu=name', '--format=csv,noheader'], capture_output=True, text=True).stdout.strip()
except Exception:
    gpu_name = ''
if SEEDS is None: SEEDS = [42, 7, 2024] if 'A100' in gpu_name else [42]
COMMIT = subprocess.check_output(['git', '-C', '/content/agri_doctor', 'rev-parse', 'HEAD']).decode().strip()
def done(name): return (OUT / name).exists()
def save(name, obj): json.dump(obj, open(OUT / name, 'w'), indent=1, default=str)
def load(name): return json.load(open(OUT / name))
print('repo', COMMIT, '| TF', tf.__version__, '| GPU', gpu_name or 'none', '| seeds', SEEDS, '| output', OUT)

GPU 0: NVIDIA A100-SXM4-80GB (UUID: GPU-c2555fdf-ea9c-53e0-54b4-132c3dc99d9d)
Mounted at /content/drive
repo ef645565c2f116d0eb7c36700d412197789d28fb | TF 2.21.0 | GPU NVIDIA A100-SXM4-80GB | seeds [42, 7, 2024] | output /content/drive/MyDrive/agri_v14


In [3]:
%%writefile /content/audit11_lib.py
"""Agri-Doctor Pro, fresh-side audit (v11).
Pure helpers used by Agri_Doctor_audit_v11.ipynb: path resolution for old records,
hash + embedding leakage detection, labelling kits, cluster-aware statistics and a
resumable Gemini runner. numpy / PIL / scipy only; torch is passed in by the notebook."""
import hashlib
import json
import math
import random
import re
import time
from collections import Counter, defaultdict
from pathlib import Path

import numpy as np
from PIL import Image, ImageDraw, ImageOps

IMG_EXT = {".jpg", ".jpeg", ".png", ".bmp", ".webp", ".tif", ".tiff"}


def norm(s):
    return re.sub(r"[^a-z0-9]+", "_", str(s).lower()).strip("_")


def list_images(root):
    return sorted(p for p in Path(root).rglob("*") if p.is_file() and p.suffix.lower() in IMG_EXT)


# ------------------------------------------------------------------ resolving old record paths
SLUG_TO_HANDLE = {
    "corn-leaf-diseasesnlb": "rabbityashow/corn-leaf-diseasesnlb",
    "paddy-doctor": "imbikramsaha/paddy-doctor",
    "paddy-leaves-8-class-dataset": "winnerkarthickm/paddy-leaves-8-class-dataset",
    "cotton-leaf-disease-dataset": "seroshkarim/cotton-leaf-disease-dataset",
    "corn-or-maize-leaf-disease-dataset": "smaranjitghose/corn-or-maize-leaf-disease-dataset",
    "cotton-plant-disease-dataset": "parthajyotiroyjoy/cotton-plant-disease-dataset",
    "rice-disease-dataset": "anshulm257/rice-disease-dataset",
    "weed-detection-in-soybean-crops": "fpeccia/weed-detection-in-soybean-crops",
    "rice-leaf-disease-image": "nirmalsankalana/rice-leaf-disease-image",
}


class Resolver:
    """Map a file path written by an earlier Colab session to the same file in this session.
    roots: {kaggle handle: local download root}. Paths under /content/... are re-created by the
    notebook at the same location, so they map to themselves."""

    def __init__(self, roots):
        self.roots = {h: Path(r) for h, r in roots.items()}
        self.miss = Counter()

    def __call__(self, p):
        p = str(p)
        m = re.match(r"^/root/\.cache/kagglehub/datasets/([^/]+)/([^/]+)/versions/\d+/(.*)$", p)
        if m:
            handle, rel = f"{m.group(1)}/{m.group(2)}", m.group(3)
        else:
            m = re.match(r"^/kaggle/input/([^/]+)/(.*)$", p)
            if m:
                handle, rel = SLUG_TO_HANDLE.get(m.group(1)), m.group(2)
            else:
                return p
        root = self.roots.get(handle)
        if root is None:
            self.miss[handle or "unknown"] += 1
            return p
        q = root / rel
        if not q.exists():                       # Colab cache vs kagglehub cache can differ by one folder level
            hits = [c / rel for c in root.iterdir() if c.is_dir() and (c / rel).exists()] if root.exists() else []
            if hits:
                q = hits[0]
        return str(q)


# ------------------------------------------------------------------ hashing (same as training/train_rice.py)
def ahash_bits(path, size=8):
    with Image.open(path) as im:
        px = np.asarray(im.convert("L").resize((size, size), Image.Resampling.BILINEAR), dtype=np.float32)
    return np.packbits(px.flatten() > px.mean())


def hash_many(paths):
    hs, ok = [], []
    for p in paths:
        try:
            hs.append(ahash_bits(p)); ok.append(str(p))
        except Exception:
            pass
    return (np.stack(hs) if hs else np.zeros((0, 8), np.uint8)), ok


_POP = np.array([bin(i).count("1") for i in range(256)], np.uint8)


def min_hamming(query_h, ref_h, chunk=512):
    """For each query hash: (minimum Hamming distance to any reference hash, index of that reference)."""
    if len(ref_h) == 0:
        return np.full(len(query_h), 64), np.full(len(query_h), -1)
    dist, idx = [], []
    for i in range(0, len(query_h), chunk):
        d = _POP[np.bitwise_xor(query_h[i:i + chunk, None, :], ref_h[None, :, :])].sum(2)
        dist.append(d.min(1)); idx.append(d.argmin(1))
    return np.concatenate(dist), np.concatenate(idx)


def hash_groups(h, threshold=4):
    """Union-find groups of hashes within `threshold` bits (same rule as the training script)."""
    n = len(h); parent = list(range(n))

    def find(i):
        while parent[i] != i:
            parent[i] = parent[parent[i]]; i = parent[i]
        return i
    for i in range(n):
        if i + 1 >= n:
            break
        d = _POP[np.bitwise_xor(h[i + 1:], h[i])].sum(1)
        for j in np.nonzero(d <= threshold)[0]:
            parent[find(i)] = find(i + 1 + int(j))
    return np.array([find(i) for i in range(n)])


# ------------------------------------------------------------------ embeddings
def dihedral(img):
    """The 8 rotations/flips of a PIL image."""
    out = []
    for flip in (False, True):
        base = ImageOps.mirror(img) if flip else img
        for k in range(4):
            out.append(base.rotate(90 * k, expand=True))
    return out


def load_rgb(path, max_side=512):
    with Image.open(path) as im:
        im = ImageOps.exif_transpose(im).convert("RGB")
        im.thumbnail((max_side, max_side))
        return im.copy()


def embed_paths(paths, forward, batch=64, views=1, log=print, workers=8):
    """forward(list of PIL images) -> (n, d) L2-normalised numpy array.
    views=8 embeds all dihedral transforms and returns (n, 8, d); unreadable files give NaN rows."""
    from concurrent.futures import ThreadPoolExecutor
    paths = [str(p) for p in paths]
    out, dim = [], None
    per = max(1, batch // views)
    with ThreadPoolExecutor(workers) as pool:
        def safe(p):
            try:
                return load_rgb(p)
            except Exception:
                return None
        for i in range(0, len(paths), per):
            imgs = list(pool.map(safe, paths[i:i + per]))
            good = [im for im in imgs if im is not None]
            flat = [v for im in good for v in (dihedral(im) if views == 8 else [im])]
            e = forward(flat) if flat else np.zeros((0, dim or 1), np.float32)
            dim = e.shape[1] if len(e) else dim
            e = e.reshape(len(good), views, -1) if views == 8 else e
            k = 0
            for im in imgs:
                if im is None:
                    out.append(np.full((views, dim or 1) if views == 8 else (dim or 1,), np.nan, np.float32))
                else:
                    out.append(e[k]); k += 1
            if (i // per) % 50 == 0:
                log(f"  embedded {min(i + per, len(paths))}/{len(paths)}")
    return np.stack(out).astype(np.float32) if out else np.zeros((0, dim or 1), np.float32)


def nearest(query, ref, chunk=2048, matmul=None):
    """Max cosine similarity of each query (n,d) or (n,8,d: max over views) to ref (m,d).
    Returns (max_sim, argmax index). matmul(a, b) can be a GPU function; default numpy."""
    mm = matmul or (lambda a, b: a @ b.T)
    ref = np.nan_to_num(ref)
    if len(ref) == 0:
        return np.full(len(query), -1.0), np.full(len(query), -1)
    sims, idx = [], []
    for i in range(0, len(query), chunk):
        q = np.nan_to_num(query[i:i + chunk])
        if q.ndim == 3:
            n, v, d = q.shape
            s = mm(q.reshape(n * v, d), ref).reshape(n, v, -1).max(1)
        else:
            s = mm(q, ref)
        sims.append(s.max(1)); idx.append(s.argmax(1))
    return np.concatenate(sims), np.concatenate(idx)


def sim_clusters(emb, threshold, chunk=2048):
    """Connected components of the graph with an edge where cosine similarity >= threshold."""
    e = np.nan_to_num(emb); n = len(e); parent = list(range(n))

    def find(i):
        while parent[i] != i:
            parent[i] = parent[parent[i]]; i = parent[i]
        return i
    for i in range(0, n, chunk):
        s = e[i:i + chunk] @ e.T
        for a, b in zip(*np.nonzero(s >= threshold)):
            a = i + int(a); b = int(b)
            if a < b:
                parent[find(a)] = find(b)
    return np.array([find(i) for i in range(n)])


# ------------------------------------------------------------------ labelling kits
SIM_BINS = [(0.98, 1.01), (0.95, 0.98), (0.90, 0.95), (0.85, 0.90), (-1.0, 0.85)]   # DINOv2 photos of one survey are often >= 0.95 apart from copies


def bin_of(s):
    for lo, hi in SIM_BINS:
        if lo <= s < hi:
            return f"{lo:.2f}-{min(hi, 1.0):.2f}" if lo >= 0 else f"<{SIM_BINS[-1][1]:.2f}"
    return f"<{SIM_BINS[-1][1]:.2f}"


def sample_pairs(cands, per_bin=25, seed=0):
    """cands: list of dicts with 'sim'. Random sample of per_bin pairs from every similarity bin,
    stratified by 'set' inside the bin so no single set dominates."""
    rng = random.Random(seed); by = defaultdict(lambda: defaultdict(list))
    for c in cands:
        by[bin_of(c["sim"])][c["set"]].append(c)
    out = []
    for b in [bin_of(lo + 1e-9) if lo >= 0 else bin_of(-0.5) for lo, _ in SIM_BINS]:
        sets = by.get(b, {})
        pools = {k: rng.sample(v, len(v)) for k, v in sorted(sets.items())}
        take = []
        while len(take) < per_bin and any(pools.values()):
            for k in sorted(pools):
                if pools[k] and len(take) < per_bin:
                    take.append(pools[k].pop())
        for t in take:
            t = dict(t); t["bin"] = b; out.append(t)
    rng.shuffle(out)                                   # labeller does not see pairs in similarity order
    for i, t in enumerate(out):
        t["pair_id"] = f"P{i:03d}"
    return out


def side_by_side(paths, captions, height=320):
    ims = []
    for p in paths:
        try:
            im = load_rgb(p, 1024)
        except Exception:
            im = Image.new("RGB", (height, height), "gray")
        w = int(im.width * height / max(1, im.height))
        ims.append(im.resize((max(1, w), height)))
    W = sum(i.width for i in ims) + 10 * (len(ims) - 1)
    canvas = Image.new("RGB", (W, height + 24), "white")
    x = 0
    d = ImageDraw.Draw(canvas)
    for im, cap in zip(ims, captions):
        canvas.paste(im, (x, 24)); d.text((x + 4, 4), str(cap)[:60], fill="black"); x += im.width + 10
    return canvas


def contact_sheet_pdf(items, out_pdf, per_page=4):
    """items: list of (title, PIL image). Writes a multi-page PDF for offline labelling."""
    pages = []
    for i in range(0, len(items), per_page):
        chunk = items[i:i + per_page]
        W = max(im.width for _, im in chunk); H = sum(im.height + 30 for _, im in chunk)
        page = Image.new("RGB", (W, H), "white"); d = ImageDraw.Draw(page); y = 0
        for title, im in chunk:
            d.text((4, y + 6), title, fill="black"); page.paste(im, (0, y + 30)); y += im.height + 30
        pages.append(page)
    if pages:
        pages[0].save(out_pdf, save_all=True, append_images=pages[1:])
    return len(pages)


def choose_threshold(labels, pairs, positive=("same_photo",), min_rate=0.9):
    """Lowest similarity bin in which at least min_rate of labelled pairs are positives,
    provided every higher bin also meets it. Returns (threshold or None, per-bin table)."""
    by = defaultdict(list)
    for p in pairs:
        lab = labels.get(p["pair_id"])
        if lab and lab != "skip":
            by[p["bin"]].append(lab in positive)
    table, thr = [], None
    for lo, hi in SIM_BINS:
        b = bin_of(lo + 1e-9) if lo >= 0 else bin_of(-0.5)
        v = by.get(b, [])
        rate = sum(v) / len(v) if v else None
        lo_ci, hi_ci = wilson(sum(v), len(v)) if v else (None, None)
        table.append({"bin": b, "labelled": len(v), "positives": int(sum(v)), "rate": rate, "ci95": [lo_ci, hi_ci]})
    for row, (lo, _) in zip(table, SIM_BINS):
        if lo < 0 or row["rate"] is None or row["rate"] < min_rate:
            break
        thr = lo
    return thr, table


# ------------------------------------------------------------------ statistics
def wilson(k, n, z=1.959964):
    if n == 0:
        return (None, None)
    p = k / n; d = 1 + z * z / n; c = (p + z * z / (2 * n)) / d
    h = z * math.sqrt(p * (1 - p) / n + z * z / (4 * n * n)) / d
    return (round(100 * (c - h), 1), round(100 * (c + h), 1))


def aurc(score, correct):
    order = np.argsort(-np.asarray(score), kind="stable")
    err = 1 - np.asarray(correct, float)[order]
    return float(np.mean(np.cumsum(err) / np.arange(1, len(err) + 1)))


def icc_binary(y, cl, min_multi=10):
    """One-way ANOVA estimator of the intra-cluster correlation for a 0/1 outcome (clipped at 0)."""
    y = np.asarray(y, float); cl = np.asarray(cl)
    groups = [y[cl == g] for g in np.unique(cl)]
    k, N = len(groups), len(y)
    if k < 2 or N <= k or sum(len(g) > 1 for g in groups) < min_multi:
        return None                                    # not estimable: too few clusters with more than one photo
    m0 = (N - sum(len(g) ** 2 for g in groups) / N) / (k - 1)
    msb = sum(len(g) * (g.mean() - y.mean()) ** 2 for g in groups) / (k - 1)
    msw = sum(((g - g.mean()) ** 2).sum() for g in groups) / (N - k)
    den = msb + (m0 - 1) * msw
    return float(max(0.0, (msb - msw) / den)) if den > 0 else 0.0


def design_effect(cl, icc):
    sizes = np.array(list(Counter(np.asarray(cl).tolist()).values()), float)
    m = sizes.mean()
    base = {"clusters": int(len(sizes)), "multi_photo_clusters": int((sizes > 1).sum()), "mean_cluster_size": round(float(m), 3),
            "max_cluster_size": int(sizes.max())}
    if icc is None:
        return {**base, "icc": None, "design_effect": None, "n_eff": None}
    deff = 1 + (m - 1) * icc
    return {**base, "icc": round(icc, 4), "design_effect": round(float(deff), 3), "n_eff": round(float(len(cl) / deff), 1)}


def cluster_boot(stat, data, cl, B=2000, seed=0):
    """Percentile CI for stat(index array) resampling whole clusters with replacement."""
    rng = np.random.default_rng(seed); cl = np.asarray(cl)
    groups = [np.nonzero(cl == g)[0] for g in np.unique(cl)]
    vals = []
    for _ in range(B):
        pick = rng.integers(0, len(groups), len(groups))
        idx = np.concatenate([groups[i] for i in pick])
        vals.append(stat(idx))
    return [round(float(np.percentile(vals, 2.5)), 4), round(float(np.percentile(vals, 97.5)), 4)]


def signflip_test(diff, cl, B=20000, seed=0):
    """Paired sign-flip permutation test on per-cluster sums of a per-image difference (two-sided)."""
    cl = np.asarray(cl); diff = np.asarray(diff, float)
    s = np.array([diff[cl == g].sum() for g in np.unique(cl)])
    obs = abs(s.sum())
    if obs == 0:
        return 1.0
    rng = np.random.default_rng(seed); hits = 0
    for start in range(0, B, 1000):
        signs = rng.choice([-1.0, 1.0], size=(min(1000, B - start), len(s)))
        hits += int((np.abs(signs @ s) >= obs - 1e-12).sum())
    return float(hits / B)


def holm(pvals):
    """Holm-Bonferroni adjusted p-values for a {name: p} dict."""
    items = sorted(pvals.items(), key=lambda kv: kv[1]); m = len(items); out, run = {}, 0.0
    for i, (k, p) in enumerate(items):
        run = max(run, min(1.0, (m - i) * p)); out[k] = round(run, 6)
    return out


def fisher_2x2(a, b, c, d):
    from scipy.stats import fisher_exact
    return float(fisher_exact([[a, b], [c, d]])[1])


# ------------------------------------------------------------------ Gemini with retries and a disk cache
NON_TRANSIENT = {"auth", "quota", "model", "missing_key"}


def image_key(image_bytes, model):
    return hashlib.sha1(image_bytes).hexdigest()[:20] + "__" + re.sub(r"[^a-zA-Z0-9.-]", "_", model)


def gemini_assess_cached(image_bytes, model, cache_dir, call, log_path=None, retries=5, base_wait=2.0, sleep=time.sleep):
    """call(image_bytes, model) -> agri_core-style dict ({'ok': True, 'assessment': ...} or {'ok': False, 'kind': ...}).
    Successful answers are cached on disk by image hash + model and never requested again.
    Transient failures are retried with exponential backoff (2, 4, 8, 16, 32 s)."""
    cache_dir = Path(cache_dir); cache_dir.mkdir(parents=True, exist_ok=True)
    f = cache_dir / (image_key(image_bytes, model) + ".json")
    if f.exists():
        return json.loads(f.read_text()), True
    out = None
    for attempt in range(retries + 1):
        out = call(image_bytes, model)
        if log_path:
            with open(log_path, "a") as lg:
                lg.write(json.dumps({"t": time.strftime("%Y-%m-%d %H:%M:%S"), "model": model, "key": f.stem,
                                     "attempt": attempt, "ok": bool(out.get("ok")), "kind": out.get("kind")}) + "\n")
        if out.get("ok"):
            f.write_text(json.dumps(out, default=str))
            return out, False
        if out.get("kind") in NON_TRANSIENT or attempt == retries:
            break
        sleep(base_wait * 2 ** attempt)
    return out, False


def vlm_table(records):
    """records: dicts with category (or None if the call failed) and correct (bool).
    Accuracy of the classifier within each agreement category, Wilson CIs, and Fisher tests."""
    cats = ["Agreement", "Partial agreement", "Disagreement", "Uncertain"]
    t = {}
    for c in cats:
        r = [x for x in records if x.get("category") == c]
        k = sum(x["correct"] for x in r)
        t[c] = {"n": len(r), "classifier_correct": k, "classifier_wrong": len(r) - k,
                "accuracy": round(100 * k / len(r), 1) if r else None, "ci95": list(wilson(k, len(r)))}
    a, d = t["Agreement"], t["Disagreement"]
    out = {"table": t, "assessed": sum(1 for x in records if x.get("category")),
           "failed": sum(1 for x in records if not x.get("category"))}
    if a["n"] and d["n"]:
        out["fisher_agreement_vs_disagreement_p"] = fisher_2x2(a["classifier_correct"], a["classifier_wrong"],
                                                               d["classifier_correct"], d["classifier_wrong"])
    rest = [t[c] for c in cats[1:]]
    rk = sum(x["classifier_correct"] for x in rest); rn = sum(x["n"] for x in rest)
    if a["n"] and rn:
        out["fisher_agreement_vs_rest_p"] = fisher_2x2(a["classifier_correct"], a["classifier_wrong"], rk, rn - rk)
    return out

Writing /content/audit11_lib.py


In [4]:
%%writefile /content/v12_lib.py
"""Agri-Doctor Pro v12: leakage-safe grouping, router calibration, risk-controlled routing thresholds,
the end-to-end routing + diagnosis pipeline, and evaluation metrics.
Builds on agri_core (the app's decision code) and audit11_lib (hash / embedding helpers)."""
import json
import math
from collections import Counter, defaultdict

import numpy as np


# ------------------------------------------------------------------ leakage-safe grouping
def merge_partitions(n, partitions, extra_edges=()):
    """Union of several partitions of n items (each an array of group labels, or None) plus explicit
    (i, j) edges. Returns one connected-component label per item."""
    from scipy.sparse import coo_matrix
    from scipy.sparse.csgraph import connected_components
    rows, cols = [], []
    for lab in partitions:
        if lab is None:
            continue
        first = {}
        for i, g in enumerate(np.asarray(lab).tolist()):
            if g in first:
                rows.append(first[g]); cols.append(i)
            else:
                first[g] = i
    for i, j in extra_edges:
        rows.append(int(i)); cols.append(int(j))
    if not rows:
        return np.arange(n)
    m = coo_matrix((np.ones(len(rows), np.int8), (rows, cols)), shape=(n, n))
    return connected_components(m, directed=False)[1]


def similarity_partition(emb, threshold, mask=None, chunk=2048, matmul=None):
    """Connected components of the graph 'cosine >= threshold' (optionally only between items where mask is True)."""
    from scipy.sparse import coo_matrix
    from scipy.sparse.csgraph import connected_components
    e = np.nan_to_num(np.asarray(emb, np.float32)); n = len(e)
    mm = matmul or (lambda a, b: a @ b.T)
    idx = np.arange(n) if mask is None else np.nonzero(mask)[0]
    sub = e[idx]; rows, cols = [], []
    for s in range(0, len(sub), chunk):
        sim = mm(sub[s:s + chunk], sub)
        a, b = np.nonzero(sim >= threshold)
        keep = (s + a) < b
        rows.append(idx[s + a[keep]]); cols.append(idx[b[keep]])
    rows = np.concatenate(rows) if rows else np.zeros(0, int); cols = np.concatenate(cols) if cols else np.zeros(0, int)
    m = coo_matrix((np.ones(len(rows), np.int8), (rows, cols)), shape=(n, n))
    return connected_components(m, directed=False)[1]


def split_by_groups(labels, groups, fractions, seed, fixed=None, kinds=None):
    """Per-class group split. fractions: {'val': 0.15, 'test': 0.15} (rest -> train).
    fixed: {index: split} assignments that win (e.g. the paper's test images); every other member
    of a group that contains a fixed item goes to 'excluded' (never trains, never tests)."""
    rng = np.random.default_rng(seed); fixed = fixed or {}
    groups = np.asarray(groups); out = {"train": [], "val": [], "test": [], "excluded": []}
    by_group = defaultdict(list)
    for i, g in enumerate(groups.tolist()):
        by_group[g].append(i)
    free_by_class = defaultdict(list)
    for g, members in by_group.items():
        fx = [i for i in members if i in fixed]
        if fx:
            for i in members:
                out[fixed[i] if i in fixed else "excluded"].append(i)
        else:
            cls = Counter(labels[i] for i in members).most_common(1)[0][0]
            kind = Counter(kinds[i] for i in members).most_common(1)[0][0] if kinds is not None else None
            free_by_class[(kind or "", cls)].append(members)
    for key in sorted(free_by_class):
        frac = fractions[key[0]] if kinds is not None else fractions
        gs = free_by_class[key]; order = rng.permutation(len(gs)); total = sum(len(g) for g in gs)
        filled = {k: 0 for k in frac}
        for o in order:
            g = gs[o]
            for k, f in frac.items():
                if filled[k] < f * total:
                    out[k].extend(g); filled[k] += len(g); break
            else:
                out["train"].extend(g)
    return {k: sorted(v) for k, v in out.items()}


# ------------------------------------------------------------------ calibration and thresholds
def temper(probs, T):
    """Temperature scaling applied to probability vectors: softmax(log p / T)."""
    lp = np.log(np.clip(np.asarray(probs, float), 1e-12, 1)) / T
    lp -= lp.max(1, keepdims=True); e = np.exp(lp)
    return e / e.sum(1, keepdims=True)


def fit_temperature(probs, y, grid=None):
    """Temperature minimising validation NLL (grid search over log-spaced T, then refine)."""
    y = np.asarray(y)
    nll = lambda T: -np.mean(np.log(np.clip(temper(probs, T)[np.arange(len(y)), y], 1e-12, 1)))
    grid = grid if grid is not None else np.exp(np.linspace(np.log(0.25), np.log(10), 60))
    best = min(grid, key=nll)
    fine = np.linspace(best * 0.85, best * 1.15, 31)
    best = min(fine, key=nll)
    return float(best), float(nll(1.0)), float(nll(best))


def ece(conf, correct, bins=15):
    conf = np.asarray(conf, float); correct = np.asarray(correct, float); e = 0.0
    edges = np.linspace(0, 1, bins + 1)
    for lo, hi in zip(edges[:-1], edges[1:]):
        m = (conf > lo) & (conf <= hi)
        if m.any():
            e += m.mean() * abs(correct[m].mean() - conf[m].mean())
    return float(e)


def cp_upper(k, n, delta=0.05):
    """One-sided Clopper-Pearson upper bound on a binomial rate."""
    from scipy.stats import beta
    if n == 0:
        return 1.0
    return 1.0 if k >= n else float(beta.ppf(1 - delta, k + 1, n - k))


def risk_thresholds(conf, pred, true, classes, alpha, delta=0.05, min_n=30, grid=None):
    """For each predicted class c: the smallest confidence threshold t such that, on validation images
    predicted c with confidence >= t, the 95% upper bound of the wrong-crop rate is <= alpha and at least
    min_n images remain. None means: no threshold certifies alpha, so c is always sent to review."""
    conf = np.asarray(conf, float); pred = np.asarray(pred); true = np.asarray(true)
    grid = grid if grid is not None else np.round(np.arange(0.50, 1.0, 0.005), 3)
    out = {}
    for ci, c in enumerate(classes):
        m = pred == ci; best = None
        for t in grid:
            sel = m & (conf >= t); n = int(sel.sum())
            if n < min_n:
                break
            k = int((true[sel] != ci).sum())
            if cp_upper(k, n, delta) <= alpha:
                best = {"threshold": float(t), "n_val": n, "wrong_val": k, "ucb": round(cp_upper(k, n, delta), 4),
                        "val_coverage_of_class": round(n / max(1, int(m.sum())), 4)}
                break
        out[c] = best
    return out


# ------------------------------------------------------------------ routing decision and pipeline
def route_v12(raw_probs, classes, cfg):
    """cfg: {'temperature': T, 'thresholds': {crop: {'threshold': t} or None}, 'min_margin': m}.
    Auto-accept only when the calibrated confidence passes that crop's validated threshold and the
    margin to the second crop is large enough; otherwise the user must confirm the crop."""
    p = temper(np.asarray(raw_probs, float)[None], cfg["temperature"])[0]
    order = np.argsort(p)[::-1]; crop = classes[int(order[0])]
    conf = float(p[order[0]]); margin = float(p[order[0]] - p[order[1]]) if len(p) > 1 else 1.0
    th = cfg["thresholds"].get(crop)
    if th is None:
        status, reason = "review", f"No validated auto-accept threshold for {crop}."
    elif conf < th["threshold"]:
        status, reason = "review", f"Calibrated confidence {conf:.3f} below the validated {crop} threshold {th['threshold']:.3f}."
    elif margin < cfg.get("min_margin", 0.0):
        status, reason = "review", f"Margin to second crop {margin:.3f} below {cfg['min_margin']:.3f}."
    else:
        status, reason = "auto", None
    return {"crop": crop, "conf": conf, "margin": margin, "status": status, "reason": reason,
            "probs_calibrated": [round(float(x), 6) for x in p], "probs_raw": [round(float(x), 6) for x in raw_probs]}


def diagnose(ac, img, interp, classes, crop):
    """The app's own diagnosis: whole-leaf verdict, quadrant corroboration, review flag."""
    whole = ac.classify_image(img, interp, classes, crop)
    quads = ac.analyze_quadrants(img, interp, classes, crop)
    s = ac.summarize_diagnosis(whole, quads)
    return {"label": whole["label"], "conf": whole["conf"], "top": [list(t) for t in whole["top"]],
            "agreement": s["agreement"], "needs_review": s["needs_review"], "verdict_kind": s["verdict_kind"],
            "review_reasons": s["review_reasons"]}


def final_decision(route, diag):
    """accepted: the crop was auto-accepted and the diagnosis needs no review.
    review_crop: the user must confirm the crop first. review_diagnosis: crop accepted, diagnosis uncertain."""
    if route["status"] != "auto":
        return "review_crop"
    return "review_diagnosis" if diag["needs_review"] else "accepted"


# ------------------------------------------------------------------ metrics
def wilson(k, n, z=1.959964):
    if n == 0:
        return [None, None]
    p = k / n; d = 1 + z * z / n; c = (p + z * z / (2 * n)) / d
    h = z * math.sqrt(p * (1 - p) / n + z * z / (4 * n * n)) / d
    return [round(100 * (c - h), 1), round(100 * (c + h), 1)]


def rate(k, n):
    return {"k": int(k), "n": int(n), "pct": round(100 * k / n, 2) if n else None, "ci95": wilson(k, n)}


def prf(true, pred, labels=None):
    labels = labels or sorted(set(true) | set(pred))
    conf = {t: {p: 0 for p in labels} for t in labels}
    for t, p in zip(true, pred):
        if t in conf and p in conf[t]:
            conf[t][p] += 1
    per = {}
    for c in labels:
        tp = conf[c][c]; fp = sum(conf[t][c] for t in labels if t != c); fn = sum(conf[c][p] for p in labels if p != c)
        if tp + fn == 0 and tp + fp == 0:
            continue
        pr = tp / (tp + fp) if tp + fp else 0.0; rc = tp / (tp + fn) if tp + fn else 0.0
        per[c] = {"precision": round(pr, 4), "recall": round(rc, 4), "f1": round(2 * pr * rc / (pr + rc), 4) if pr + rc else 0.0,
                  "support": tp + fn}
    present = [c for c in per if per[c]["support"] > 0]
    macro = round(float(np.mean([per[c]["f1"] for c in present])), 4) if present else None
    acc = sum(t == p for t, p in zip(true, pred)) / len(true) if true else None
    return {"accuracy": round(100 * acc, 2) if acc is not None else None, "macro_f1_present_classes": macro,
            "per_class": per, "confusion": conf, "labels": labels}


def routing_metrics(records):
    """records: dicts with true_crop, crop, status. Wrong-crop auto-accept is the safety metric."""
    n = len(records)
    correct = sum(r["crop"] == r["true_crop"] for r in records)
    auto = [r for r in records if r["status"] == "auto"]
    fa = sum(r["crop"] != r["true_crop"] for r in auto)
    return {"n": n, "routing_accuracy": rate(correct, n), "auto_accept_rate": rate(len(auto), n),
            "false_auto_accepts": fa, "false_auto_accept_rate": rate(fa, n),
            "false_auto_accept_among_auto": rate(fa, len(auto)),
            "routed_to": dict(Counter(r["crop"] for r in records))}


def end_to_end_metrics(records):
    """records: dicts with true, true_crop, crop, status, final, pred (diagnosis with the routed crop's model),
    pred_confirmed (diagnosis with the true crop's model, i.e. after the user confirms the crop)."""
    n = len(records)
    acc_auto = [r for r in records if r["final"] == "accepted"]
    ok_accepted = sum(r["crop"] == r["true_crop"] and r["pred"] == r["true"] for r in acc_auto)
    with_conf = sum((r["pred"] if (r["status"] == "auto") else r["pred_confirmed"]) == r["true"] and
                    (r["status"] != "auto" or r["crop"] == r["true_crop"]) for r in records)
    return {"n": n,
            "accepted_without_review": rate(len(acc_auto), n),
            "accuracy_of_accepted": rate(ok_accepted, len(acc_auto)),
            "wrong_but_accepted": rate(len(acc_auto) - ok_accepted, n),
            "accuracy_after_crop_confirmation": rate(with_conf, n),
            "outcomes": dict(Counter(r["final"] for r in records))}


# ------------------------------------------------------------------ per-image replay (same record format as the paper's runs)
def old_rule(ac, quads, healthy_label):
    bad = [r["label"] for r in quads.values() if r["conf"] > ac.DETECTION_THRESHOLD and ac.label_kind(r["label"]) != "healthy"]
    return Counter(bad).most_common(1)[0][0] if bad else healthy_label


def replay_row(ac, img, true_label, true_crop, model_crop, interp, classes, router_interp=None, router_cfg=None, tag=""):
    """Everything the app computes for one photo (whole leaf, quadrants, review flag, TTA, router),
    in the record format of the earlier runs, plus the calibrated v12 routing decision."""
    from PIL import ImageOps
    img = ImageOps.exif_transpose(img).convert("RGB")
    probs = ac.predict_image(img, interp).astype(float)
    ac.check_probabilities(probs, model_crop)
    whole = ac.classify_image(img, interp, classes, model_crop)
    quads = ac.analyze_quadrants(img, interp, classes, model_crop)
    s = ac.summarize_diagnosis(whole, quads)
    tta = [ac.predict_image(t, interp).astype(float) for t in (ImageOps.mirror(img), ImageOps.flip(img))]
    healthy = next(c for c in classes if "Healthy" in c)
    row = {"tag": tag, "true": true_label, "true_crop": true_crop, "model_crop": model_crop,
           "probs": [round(float(x), 6) for x in probs], "pred": whole["label"], "conf": whole["conf"],
           "agreement": s["agreement"], "needs_review": s["needs_review"], "verdict_kind": s["verdict_kind"],
           "pred_old": old_rule(ac, quads, healthy), "tta_probs": [[round(float(x), 6) for x in t] for t in tta],
           "quad_labels": [q["label"] for q in quads.values()]}
    if router_interp is not None:
        rp = ac.predict_image(img, router_interp).astype(float); rp = rp / rp.sum()
        r = ac.route_crop(rp)
        row.update({"router_crop": r["crop"], "router_conf": r["confidence"], "router_review": r["needs_review"]})
        if router_cfg is not None:
            v = route_v12(rp, list(ac.ROUTER_CLASSES), router_cfg)
            row.update({"router_v12_crop": v["crop"], "router_v12_conf": v["conf"], "router_v12_status": v["status"],
                        "router_probs": [round(float(x), 6) for x in rp]})
    return row

Writing /content/v12_lib.py


In [5]:
%%writefile /content/v14_train.py
"""Agri-Doctor Pro v14 rice training: EfficientNetV2-S at 300 px, full fine-tuning with cosine decay,
label smoothing, heavy augmentation, class weights, early stopping on validation accuracy.
The exported TFLite model takes raw 0-255 RGB (preprocessing built in), like every model in the app."""
import json
import math
from collections import Counter
from pathlib import Path

import numpy as np


def class_weights(labels, classes):
    c = Counter(labels); n = len(labels); k = len([x for x in classes if c[x]])
    return {i: (n / (k * c[x]) if c[x] else 0.0) for i, x in enumerate(classes)}


def make_ds(tf, paths, ys, n_classes, size, batch, shuffle, seed, smoothing=0.0):
    def load(p, y):
        img = tf.io.decode_image(tf.io.read_file(p), channels=3, expand_animations=False)
        img = tf.image.resize(img, (size, size))
        oh = tf.one_hot(y, n_classes)
        if smoothing:
            oh = oh * (1 - smoothing) + smoothing / n_classes
        return tf.cast(img, tf.float32), oh
    ds = tf.data.Dataset.from_tensor_slices((list(map(str, paths)), list(ys)))
    if shuffle:
        ds = ds.shuffle(len(paths), seed=seed, reshuffle_each_iteration=True)
    return ds.map(load, num_parallel_calls=tf.data.AUTOTUNE).batch(batch).prefetch(tf.data.AUTOTUNE)


def build(tf, n_classes, size, backbone_name="EfficientNetV2S", weights="imagenet", dropout=0.3):
    L = tf.keras.layers
    augment = tf.keras.Sequential([
        L.RandomFlip("horizontal_and_vertical"), L.RandomRotation(0.15), L.RandomZoom((-0.25, 0.15)),
        L.RandomTranslation(0.1, 0.1), L.RandomContrast(0.3), L.RandomBrightness(0.25, value_range=(0, 255))], name="augment")
    backbone = getattr(tf.keras.applications, backbone_name)(include_top=False, weights=weights, input_shape=(size, size, 3),
                                                            include_preprocessing=True, pooling="avg")
    drop = L.Dropout(dropout); head = L.Dense(n_classes, activation="softmax", dtype="float32", name="head")
    inp = L.Input((size, size, 3))
    train_model = tf.keras.Model(inp, head(drop(backbone(augment(inp)))), name="rice_v14_train")
    inp2 = L.Input((size, size, 3))
    infer_model = tf.keras.Model(inp2, head(drop(backbone(inp2, training=False), training=False)), name="rice_v14")
    return train_model, infer_model, backbone


def train(tf, cfg, train_items, val_items, classes, out_dir, log=print):
    """cfg: size, batch, seed, epochs_head, epochs_full, lr_head, lr_full, patience, smoothing, backbone, weights."""
    out_dir = Path(out_dir); out_dir.mkdir(parents=True, exist_ok=True)
    tf.keras.utils.set_random_seed(cfg["seed"])
    idx = {c: i for i, c in enumerate(classes)}
    tp, ty = [p for p, _ in train_items], [idx[l] for _, l in train_items]
    vp, vy = [p for p, _ in val_items], [idx[l] for _, l in val_items]
    tr_ds = make_ds(tf, tp, ty, len(classes), cfg["size"], cfg["batch"], True, cfg["seed"], cfg["smoothing"])
    va_ds = make_ds(tf, vp, vy, len(classes), cfg["size"], cfg["batch"], False, cfg["seed"])
    model, infer, backbone = build(tf, len(classes), cfg["size"], cfg["backbone"], cfg["weights"])
    cw = class_weights([l for _, l in train_items], classes)
    loss = tf.keras.losses.CategoricalCrossentropy()
    ck = out_dir / "best.weights.h5"
    def callbacks():
        return [tf.keras.callbacks.ModelCheckpoint(str(ck), monitor="val_accuracy", save_best_only=True, save_weights_only=True),
                tf.keras.callbacks.EarlyStopping(monitor="val_accuracy", patience=cfg["patience"], restore_best_weights=True)]
    hist = {}
    backbone.trainable = False                                       # 1) head on a frozen backbone
    model.compile(tf.keras.optimizers.Adam(cfg["lr_head"]), loss, metrics=["accuracy"])
    h = model.fit(tr_ds, validation_data=va_ds, epochs=cfg["epochs_head"], class_weight=cw, callbacks=callbacks(), verbose=2)
    hist["head"] = {k: [float(x) for x in v] for k, v in h.history.items()}
    backbone.trainable = True                                        # 2) every layer, cosine-decayed learning rate
    for layer in backbone.layers:
        if isinstance(layer, tf.keras.layers.BatchNormalization):
            layer.trainable = False                                  # keep BN statistics stable with small batches
    steps = max(1, math.ceil(len(tp) / cfg["batch"])) * cfg["epochs_full"]
    sched = tf.keras.optimizers.schedules.CosineDecay(cfg["lr_full"], steps, alpha=0.01)
    model.compile(tf.keras.optimizers.AdamW(sched, weight_decay=1e-5), loss, metrics=["accuracy"])
    h = model.fit(tr_ds, validation_data=va_ds, epochs=cfg["epochs_full"], class_weight=cw, callbacks=callbacks(), verbose=2)
    hist["full"] = {k: [float(x) for x in v] for k, v in h.history.items()}
    if ck.exists():
        model.load_weights(str(ck))
    infer.compile(loss=loss, metrics=["accuracy"])
    val_acc = float(infer.evaluate(va_ds, verbose=0)[1])
    infer.save(out_dir / "rice_model_v14.keras")
    conv = tf.lite.TFLiteConverter.from_keras_model(infer)
    conv.optimizations = [tf.lite.Optimize.DEFAULT]; conv.target_spec.supported_types = [tf.float16]   # half-size file, same accuracy
    (out_dir / "rice_model.tflite").write_bytes(conv.convert())
    (out_dir / "labels.txt").write_text("\n".join(classes) + "\n")
    log(f"seed {cfg['seed']}: best validation accuracy {val_acc:.4f}")
    return {"val_accuracy": val_acc, "history": hist, "class_weights": {classes[i]: w for i, w in cw.items()}, "cfg": cfg,
            "train_images": len(tp), "val_images": len(vp)}

Writing /content/v14_train.py


In [6]:
# 2. Repository inspection: what exists, what is reused, what is missing
import agri_core as ac, train_rice as tr, audit11_lib as al, v12_lib as v12
app_src = Path('/content/agri_doctor/app.py').read_text()
IDS = dict(re.findall(r'^(ROUTER|CORN|RICE|COTTON)_MODEL_ID\s*=\s*"([^"]+)"', app_src, re.M))
repo_files = sorted(str(p.relative_to('/content/agri_doctor')) for p in Path('/content/agri_doctor').rglob('*')
                    if p.is_file() and '.git/' not in str(p) and 'images/' not in str(p))
inventory = {
    'repo_commit': COMMIT,
    'reused': {'agri_core.py': 'class lists, preprocessing (raw 0-255), predict/classify, quadrant analysis, summarize_diagnosis, route_crop',
               'training/train_rice.py': 'EfficientNetV2-B0 recipe (build_model, train, export_tflite, tflite_predictor, evaluate)',
               'app.py': 'Google Drive IDs of the deployed router, maize, rice and cotton TFLite models'},
    'classes': {'router': ac.ROUTER_CLASSES, 'maize': ac.CORN_CLASSES, 'rice': ac.RICE_CLASSES, 'cotton': ac.COTTON_CLASSES},
    'router_rule': {'ROUTER_THRESHOLD': ac.ROUTER_THRESHOLD, 'ROUTER_CONFUSABLE': sorted(ac.ROUTER_CONFUSABLE)},
    'deployed_model_ids': IDS,
    'training_notebooks_in_repo': [f for f in repo_files if f.endswith('.ipynb')],
    'missing_from_repo': ['Keras checkpoints of the deployed maize/cotton/rice models (only TFLite files are published via Drive IDs)',
                          'training code of the deployed router (crop_router_v2.tflite)', 'any farm, plot or date metadata for field photos'],
    'transfer_learning_decision': ('The maize and cotton models use the same ImageNet EfficientNetV2-B0 backbone as train_rice.py, but only '
                                   'their TFLite exports are available, which cannot be fine-tuned. The rice model is therefore retrained '
                                   'from the ImageNet backbone with the repository recipe; no crop model is used as initialisation.')}
assert set(IDS) == {'ROUTER', 'CORN', 'RICE', 'COTTON'}, f'model IDs not found in app.py: {IDS}'
save('inventory.json', inventory); print(json.dumps(inventory, indent=1)[:2500])

{
 "repo_commit": "ef645565c2f116d0eb7c36700d412197789d28fb",
 "reused": {
  "agri_core.py": "class lists, preprocessing (raw 0-255), predict/classify, quadrant analysis, summarize_diagnosis, route_crop",
  "training/train_rice.py": "EfficientNetV2-B0 recipe (build_model, train, export_tflite, tflite_predictor, evaluate)",
  "app.py": "Google Drive IDs of the deployed router, maize, rice and cotton TFLite models"
 },
 "classes": {
  "router": [
   "Cotton",
   "Maize",
   "Rice"
  ],
  "maize": [
   "Maize_Blight",
   "Maize_Common_Rust",
   "Maize_Gray_Leaf_Spot",
   "Maize_Healthy",
   "Weed_Broadleaf",
   "Weed_Grass"
  ],
  "rice": [
   "Rice_Bacterial_Leaf_Blight",
   "Rice_Brown_Spot",
   "Rice_Healthy",
   "Rice_Leaf_Blast",
   "Rice_Leaf_scald",
   "Rice_Sheath_Blight",
   "Weed_Broadleaf",
   "Weed_Grass"
  ],
  "cotton": [
   "Disease_Bacterial_Blight",
   "Disease_Curl_Virus",
   "Disease_Healthy",
   "Disease_Herbicide_Damage",
   "Disease_Leaf_Reddening",
   "Weed_Carpetwe

In [7]:
# 3. Checkpoints: download and validate the deployed TFLite models (originals are never modified)
import gdown
DEPLOYED = {'router': ('ROUTER', ac.ROUTER_CLASSES), 'rice': ('RICE', ac.RICE_CLASSES)}
DEP_DIR = Path('/content/deployed'); DEP_DIR.mkdir(exist_ok=True)
def load_interp(path):
    try:
        from ai_edge_litert.interpreter import Interpreter
    except ImportError:
        Interpreter = tf.lite.Interpreter
    it = Interpreter(model_path=str(path), num_threads=os.cpu_count()); it.allocate_tensors(); return it
dep_info, DEP = {}, {}
for name, (key, classes) in DEPLOYED.items():
    p = DEP_DIR / f'{name}.tflite'
    if not p.exists(): gdown.download(id=IDS[key], output=str(p), quiet=True)
    ac.validate_tflite_file(p)
    DEP[name] = load_interp(p); ac.check_output_classes(DEP[name], classes, name)
    dep_info[name] = {'drive_id': IDS[key], 'bytes': p.stat().st_size, 'sha256': hashlib.sha256(p.read_bytes()).hexdigest()}
save('deployed_models.json', dep_info); print(json.dumps(dep_info, indent=1))

/usr/local/lib/python3.13/dist-packages/tensorflow/lite/python/interpreter.py:457: UserWarning:     Warning: tf.lite.Interpreter is deprecated and is scheduled for deletion in
    TF 2.20. Please use the LiteRT interpreter from the ai_edge_litert package.
    See the [migration guide](https://ai.google.dev/edge/litert/migration)
    for details.
    
  warnings.warn(_INTERPRETER_DELETION_WARNING)


{
 "router": {
  "drive_id": "10LuRi-n6wFu4um9deAWBu0sr0-8e1MLV",
  "bytes": 8880536,
  "sha256": "8fc60b520001fa1bfa161e7f71364ed65443fd638fcbaa411390d65934ea859c"
 },
 "rice": {
  "drive_id": "1p2vZgq_FBigVnlhQPLQD4w2yjDn4zus3",
  "bytes": 6915856,
  "sha256": "0bc2c3c1f761490743b447d327277bbe55bd46153c87cabd30376205872d925c"
 }
}


In [8]:
# 4. Datasets, the paper's test lists, label mapping and validation
import kagglehub
from PIL import Image, ImageOps
from concurrent.futures import ThreadPoolExecutor
NEEDED = {'agri_results_v2': 'rows_rice_field__rice_new.json', 'agri_results_followup': 'field_split.json',
          'agri_results_followup2': 'rows_rice_paddydoctor__adapted_seed42.json', 'agri_results_final': 'paddydoctor_pool.json'}
REC_ROOT = Path('/content/records'); REC_ROOT.mkdir(exist_ok=True)
def find_dir(marker):
    hits = sorted(REC_ROOT.rglob(marker)); return hits[0].parent if hits else None
for name, marker in NEEDED.items():
    if find_dir(marker): continue
    for cand in [BASE / 'agri_audit_inputs' / f'{name}.zip', Path('/content/agri_doctor') / f'{name}.zip', BASE / f'{name}.zip']:
        if cand.exists():
            with zipfile.ZipFile(cand) as z:
                for m in z.namelist():
                    if m.endswith('.json'): z.extract(m, REC_ROOT / name)
            break
REC = {n: find_dir(m) for n, m in NEEDED.items()}
missing = [n for n, d in REC.items() if not d]
if missing:
    raise SystemExit(f'MISSING result archives {missing}: put them in My Drive/agri_audit_inputs/ (they define the paper test sets).')
ROOTS = {}
for h in sorted(set(al.SLUG_TO_HANDLE.values())):
    try: ROOTS[h] = Path(kagglehub.dataset_download(h))
    except Exception as e: raise SystemExit(f'Kaggle download failed for {h}: {str(e)[:200]}. Add KAGGLE_USERNAME/KAGGLE_KEY secrets.')
if not Path('/content/PlantDoc').exists():
    !git clone -q --depth 1 https://github.com/pratikkayal/PlantDoc-Dataset /content/PlantDoc
SETHY = Path('/content/sethy_field')
if not SETHY.exists():
    src = ROOTS['nirmalsankalana/rice-leaf-disease-image']
    smap = [(('tungro',), None), (('bacterial',), 'Rice_Bacterial_Leaf_Blight'), (('blight',), 'Rice_Bacterial_Leaf_Blight'),
            (('blast',), 'Rice_Leaf_Blast'), (('brown',), 'Rice_Brown_Spot')]
    for f in sorted(src.rglob('*')):
        if not f.is_file() or f.suffix.lower() not in al.IMG_EXT: continue
        rel = al.norm(f.parent.relative_to(src)); target = 'UNMATCHED'
        for keys, nm in smap:
            if all(k in rel for k in keys): target = nm; break
        if target in (None, 'UNMATCHED'): continue
        (SETHY / target).mkdir(parents=True, exist_ok=True); shutil.copy2(f, SETHY / target / f'{al.norm(f.parent.name)}__{f.name}')
R = al.Resolver(ROOTS)
def collect_by_leaf(root, exact, keywords=()):
    items = []
    for p in al.list_images(root):
        name = al.norm(p.parent.name); lab = exact.get(name, 'UNMATCHED')
        if name not in exact:
            for keys, l in keywords:
                if all(k in name for k in keys): lab = l; break
        if lab not in (None, 'UNMATCHED'): items.append((str(p), lab))
    return items
BLB, BS, HL, LB = 'Rice_Bacterial_Leaf_Blight', 'Rice_Brown_Spot', 'Rice_Healthy', 'Rice_Leaf_Blast'
# laboratory collections, mapped to the app's class names exactly as in the paper's notebooks
rice_data = ROOTS['anshulm257/rice-disease-dataset']
while len([p for p in rice_data.iterdir() if p.is_dir()]) == 1: rice_data = [p for p in rice_data.iterdir() if p.is_dir()][0]
targets = {al.norm(c.replace('Rice_', '')): c for c in ac.RICE_CLASSES if c.startswith('Rice_')}; targets['healthy_rice_leaf'] = 'Rice_Healthy'
rice_lab = [(str(p), targets[al.norm(f.name)]) for f in sorted(x for x in rice_data.iterdir() if x.is_dir()) if al.norm(f.name) in targets for p in al.list_images(f)]
skipped = [f.name for f in rice_data.iterdir() if f.is_dir() and al.norm(f.name) not in targets]
soy = ROOTS['fpeccia/weed-detection-in-soybean-crops'] / 'dataset'
rice_lab += [(str(p), lab) for s_, lab in (('broadleaf', 'Weed_Broadleaf'), ('grass', 'Weed_Grass')) for p in al.list_images(soy / s_)]
FIELD = {
    'sethy': [(str(p), p.parent.name) for p in al.list_images(SETHY)],
    'paddydoctor': [(p, l) for p, l in collect_by_leaf(ROOTS['imbikramsaha/paddy-doctor'], {'bacterial_leaf_blight': BLB, 'blast': LB, 'brown_spot': BS,
                    'normal': HL, 'bacterial_leaf_streak': None, 'bacterial_panicle_blight': None, 'dead_heart': None, 'downy_mildew': None,
                    'hispa': None, 'tungro': None}) if 'train_images' in al.norm(p)],
}
# optional extra field datasets (Mendeley zips in My Drive/agri_extra_rice/)
def extract_all(zp, dest):
    dest.mkdir(parents=True, exist_ok=True)
    if not any(dest.iterdir()):
        with zipfile.ZipFile(zp) as z: z.extractall(dest)
    for _ in range(3):                                            # nested archives
        for f in list(dest.rglob('*')):
            s = f.suffix.lower()
            if s == '.zip':
                with zipfile.ZipFile(f) as z: z.extractall(f.with_suffix(''))
                f.unlink()
            elif s in ('.rar', '.7z'):
                subprocess.run(['7z', 'x', '-y', f'-o{f.with_suffix("")}', str(f)], stdout=subprocess.DEVNULL); f.unlink()
EXTRA_RULES = [(('tungro',), None), (('turngo',), None), (('narrow',), None), (('smut',), None), (('hispa',), None),
               (('bacterial',), BLB), (('blight', 'leaf'), BLB), (('brown',), BS), (('blast',), LB), (('sheath',), 'Rice_Sheath_Blight'),
               (('steath',), 'Rice_Sheath_Blight'), (('scald',), 'Rice_Leaf_scald'), (('scaled',), 'Rice_Leaf_scald'),
               (('healthy',), HL), (('normal',), HL)]
def extra_items(root, require=None, exclude=()):
    items, unmatched = [], Counter()
    for p in al.list_images(root):
        rel = al.norm(str(p.relative_to(root)))
        if require and require not in rel: continue
        if any(x in rel for x in exclude): continue
        lab = 'UNMATCHED'
        for keys, l in EXTRA_RULES:
            if all(k in rel for k in keys): lab = l; break
        if lab == 'UNMATCHED': unmatched[p.parent.name] += 1
        elif lab: items.append((str(p), lab))
    return items, dict(unmatched)
EXTRA_SPECS = {'riceleafbd': (('riceleafbd', 'kx9rx8p2mz'), None, ()), 'bd5': (('bd5', 'sjkptpgv79', 'five_class'), None, ()),
               'dhan': (('dhan', 'shomadhan', 'znsxdctwtt'), 'field', ('white',))}
EXTRA = {}
!apt-get -qq install -y p7zip-full > /dev/null 2>&1 || true
for zp in sorted((BASE / EXTRA_DIR_NAME).glob('*.zip')) if (BASE / EXTRA_DIR_NAME).exists() else []:
    key = next((k for k, (names, _, _) in EXTRA_SPECS.items() if any(n in al.norm(zp.name) for n in names)), None)
    if key is None:
        print('skipped (not recognised, rename it to include riceleafbd / bd5 / dhan):', zp.name); continue
    dest = Path('/content/extra') / key; extract_all(zp, dest)
    its, unm = extra_items(dest, EXTRA_SPECS[key][1], EXTRA_SPECS[key][2])
    EXTRA[key] = its; print(f'{key}: {len(its)} photos {dict(Counter(l for _, l in its))}; unmatched folders {unm}')
    if not its: print(f'  WARNING {key}: no photos matched - send me the folder names above')
FIELD.update(EXTRA)
print('extra field datasets:', {k: len(v) for k, v in EXTRA.items()} or 'none found in My Drive/' + EXTRA_DIR_NAME)
J = lambda d, f: json.load(open(REC[d] / f))
fu2_tests = J('agri_results_followup2', 'test_sets.json')
TESTS = {   # the paper's held-out test photos: (path, label), crop, field source it belongs to
    'rice_sethy': ([(R(p), l) for p, l in J('agri_results_followup', 'field_split.json')['test']], 'Rice', 'sethy'),
    'rice_paddydoctor': ([(R(p), l) for p, l in fu2_tests['rice_paddydoctor']], 'Rice', 'paddydoctor'),
    'rice_trichy': ([(R(p), l) for p, l in fu2_tests['rice_trichy']], 'Rice', None),
}
def readable(items):
    def ok(p):
        try:
            with Image.open(p) as im: im.verify()
            return True
        except Exception: return False
    with ThreadPoolExecutor(16) as ex_: flags = list(ex_.map(ok, [p for p, _ in items]))
    return [it for it, f in zip(items, flags) if f], sum(not f for f in flags)
report = {}
for name in ('rice_lab',):
    globals()[name], bad = readable(globals()[name]); report[name] = {'n': len(globals()[name]), 'unreadable': bad, 'classes': dict(Counter(l for _, l in globals()[name]))}
for k in FIELD:
    FIELD[k], bad = readable(FIELD[k]); report[k] = {'n': len(FIELD[k]), 'unreadable': bad, 'classes': dict(Counter(l for _, l in FIELD[k]))}
for k, (items, crop, _) in TESTS.items():
    found = [(p, l) for p, l in items if Path(p).exists()]
    report[f'test:{k}'] = {'paper_n': len(items), 'found_now': len(found)}
    TESTS[k] = (found, crop, TESTS[k][2])
report['rice_lab_skipped_folders'] = skipped
save('data_report.json', report); print(json.dumps(report, indent=1)[:3000])
assert all(report[n]['n'] > 0 for n in ('rice_lab', 'sethy', 'paddydoctor')), 'an empty data source - see data_report.json'
if CROSS_SOURCE_HOLDOUT not in EXTRA: CROSS_SOURCE_HOLDOUT = None
assert set(l for _, l in rice_lab) == set(ac.RICE_CLASSES), f'rice lab classes {set(l for _, l in rice_lab)} != app classes'
for k, (items, _, _) in TESTS.items():
    if len(items) < 0.98 * report[f'test:{k}']['paper_n']: print(f'WARNING {k}: only {len(items)} of {report[f"test:{k}"]["paper_n"]} paper test photos found again')

100%|██████████| 0.99G/0.99G [00:25<00:00, 42.3MB/s]

Extracting files...


100%|██████████| 2.37G/2.37G [01:01<00:00, 41.4MB/s]

Extracting files...


100%|██████████| 1.02G/1.02G [00:25<00:00, 42.4MB/s]

...

100%|██████████| 161M/161M [00:04<00:00, 37.8MB/s]

Extracting files...


100%|██████████| 4.65G/4.65G [01:57<00:00, 42.4MB/s]

Extracting files...


bd5: 2180 photos {'Rice_Leaf_Blast': 720, 'Rice_Healthy': 834, 'Rice_Bacterial_Leaf_Blight': 626}; unmatched folders {}
riceleafbd: 1030 photos {'Rice_Bacterial_Leaf_Blight': 422, 'Rice_Brown_Spot': 356, 'Rice_Healthy': 252}; unmatched folders {}
dhan: 212 photos {'Rice_Leaf_scald': 74, 'Rice_Leaf_Blast': 74, 'Rice_Sheath_Blight': 64}; unmatched folders {'Browon Spot': 49, 'Rice Turgro': 76}
extra field datasets: {'bd5': 2180, 'riceleafbd': 1030, 'dhan': 212}
{
 "rice_lab": {
  "n": 8540,
  "unreadable": 0,
  "classes": {
   "Rice_Bacterial_Leaf_Blight": 636,
   "Rice_Brown_Spot": 646,
   "Rice_Healthy": 653,
   "Rice_Leaf_Blast": 634,
   "Rice_Leaf_scald": 628,
   "Rice_Sheath_Blight": 632,
   "Weed_Broadleaf": 1191,
   "Weed_Grass": 3520
  }
 },
 "sethy": {
  "n": 4624,
  "unreadable": 0,
  "classes": {
   "Rice_Bacterial_Leaf_Blight": 1584,
   "Rice_Brown_Spot": 1600,
   "Rice_Leaf_Blast": 1440
  }
 },
 "paddydoctor": {
  "n": 4946,
  "unreadable": 0,
  "classes": {
   "Rice_Bacteri

In [9]:
# 5. Split: the paper's test photos (and exact hash copies of them) are held out; everything else trains (85%) or validates (15%)
from concurrent.futures import ThreadPoolExecutor
def hash_parallel(paths):
    def one(p):
        try: return al.ahash_bits(p)
        except Exception: return np.zeros(8, np.uint8)
    with ThreadPoolExecutor(16) as ex_: return np.stack(list(ex_.map(one, paths)))
DOMAINS = {'rice': {'lab': rice_lab, 'field': ['sethy', 'paddydoctor'] + sorted(EXTRA), 'tests': ['rice_sethy', 'rice_paddydoctor']}}
SPLITS, EXTRA_TESTS = {}, {}
for dom, spec in DOMAINS.items():
    f = OUT / f'split_{dom}.json'
    if f.exists(): SPLITS[dom] = json.load(open(f)); continue
    items, kinds, sources = [], [], []
    for p, l in spec['lab']: items.append((p, l)); kinds.append('lab'); sources.append('lab')
    for s in spec['field']:
        for p, l in FIELD[s]: items.append((p, l)); kinds.append('field'); sources.append(s)
    index = {p: i for i, (p, _) in enumerate(items)}; test_idx = set()
    for t_ in spec['tests']:
        for p, l in TESTS[t_][0]:
            if p not in index: index[p] = len(items); items.append((p, l)); kinds.append('field'); sources.append('test_only')
            test_idx.add(index[p])
    n = len(items); t0 = time.time()
    H = hash_parallel([p for p, _ in items]); groups = al.hash_groups(H, HASH_THRESHOLD)
    for src in sorted(EXTRA):                                   # hold out a share of every new dataset, whole hash groups only
        idx = [i for i in range(n) if sources[i] == src]; rng = random.Random(0); by = defaultdict(list)
        for i in idx: by[(items[i][1], int(groups[i]))].append(i)
        per_class = defaultdict(list)
        for (lab, g), mem in by.items(): per_class[lab].append(mem)
        chosen = []
        for lab, gs in sorted(per_class.items()):
            rng.shuffle(gs); target = EXTRA_TEST_FRACTION * sum(len(g) for g in gs); got = 0
            for g in gs:
                if got >= target: break
                chosen += g; got += len(g)
        test_idx |= set(chosen); EXTRA_TESTS[f'rice_{src}'] = sorted(chosen)
    test_groups = {groups[i] for i in test_idx}
    excl = {i for i in range(n) if i not in test_idx and groups[i] in test_groups}      # exact / hash copies of a test photo
    rest = [i for i in range(n) if i not in test_idx and i not in excl]
    sp_rest = v12.split_by_groups([items[i][1] for i in rest], groups[rest], {'lab': LAB_FRACTIONS, 'field': FIELD_FRACTIONS}, 0,
                                  kinds=[kinds[i] for i in rest])
    sp = {k: sorted(rest[j] for j in v) for k, v in sp_rest.items()}
    sp['test'] = sorted(sp.get('test', []) + sorted(test_idx)); sp['excluded'] = sorted(excl)
    SPLITS[dom] = {'items': items, 'kinds': kinds, 'sources': sources, 'groups': groups.tolist(), 'split': sp,
                   'thresholds': {'hash': HASH_THRESHOLD, 'rule': 'test photos and their hash copies held out; no embedding filter'}}
    SPLITS[dom]['extra_tests'] = EXTRA_TESTS
    json.dump(SPLITS[dom], open(f, 'w')); print(f'{dom}: {n} images split in {(time.time() - t0) / 60:.1f} min')
S = SPLITS['rice']; EXTRA_TESTS = S.get('extra_tests', {})
for k, idx in EXTRA_TESTS.items(): TESTS[k] = ([tuple(S['items'][i]) for i in idx], 'Rice', k[5:])
rep = {}; sp = S['split']; tr_, va_, ex_ = set(sp['train']), set(sp['val']), set(sp['excluded'])
for src in ['lab', 'sethy', 'paddydoctor'] + sorted(EXTRA):
    idx = [i for i, s in enumerate(S['sources']) if s == src]
    rep[src] = {'photos': len(idx), 'train': sum(i in tr_ for i in idx), 'val': sum(i in va_ for i in idx), 'held_out_hash_copy_of_test': sum(i in ex_ for i in idx), 'test': sum(i in set(sp['test']) for i in idx)}
THRESHOLD_SOURCE = 'hash only (paper rule)'
save('leakage_split_report.json', rep); print(json.dumps(rep, indent=1))

rice: 19352 images split in 0.6 min
{
 "lab": {
  "photos": 8540,
  "train": 4347,
  "val": 944,
  "held_out_hash_copy_of_test": 2315,
  "test": 934
 },
 "sethy": {
  "photos": 4624,
  "train": 2296,
  "val": 406,
  "held_out_hash_copy_of_test": 1019,
  "test": 903
 },
 "paddydoctor": {
  "photos": 4946,
  "train": 1101,
  "val": 194,
  "held_out_hash_copy_of_test": 3051,
  "test": 600
 },
 "bd5": {
  "photos": 0,
  "train": 0,
  "val": 0,
  "held_out_hash_copy_of_test": 0,
  "test": 0
 },
 "dhan": {
  "photos": 212,
  "train": 102,
  "val": 24,
  "held_out_hash_copy_of_test": 31,
  "test": 55
 },
 "riceleafbd": {
  "photos": 1030,
  "train": 596,
  "val": 98,
  "held_out_hash_copy_of_test": 113,
  "test": 223
 }
}


In [10]:
# 6. Rice model v14: EfficientNetV2-S 300 px, full fine-tuning, augmentation, label smoothing, class weights, 3 seeds
import v14_train as vt
if GPU: tf.keras.mixed_precision.set_global_policy('mixed_float16')
CACHE = Path('/content/train_cache'); CACHE.mkdir(exist_ok=True)
def shrink(p):
    q = CACHE / (hashlib.sha1(p.encode()).hexdigest() + '.jpg')
    if not q.exists():
        with Image.open(p) as im:
            im = ImageOps.exif_transpose(im).convert('RGB'); im.thumbnail((448, 448)); im.save(q, quality=92)
    return str(q)
def part(S, name, drop_sources=()):
    its = [S['items'][i] for i in S['split'][name] if S['sources'][i] not in drop_sources and (name != 'test' or S['kinds'][i] == 'lab')]
    with ThreadPoolExecutor(16) as ex_: small = list(ex_.map(shrink, [p for p, _ in its]))
    return [(q, l) for q, (_, l) in zip(small, its)]
def lab_test_acc(path, items):
    it = load_interp(path); ok = 0
    for p, l in items:
        with Image.open(p) as im: ok += ac.classify_image(im.convert('RGB'), it, ac.RICE_CLASSES, 'Rice')['label'] == l
    return ok / max(1, len(items))
S = SPLITS['rice']; LAB_TEST = part(S, 'test')
def run_one(key, seed, drop=()):
    run = MODELS / key
    if (run / 'metrics.json').exists(): print(key, 'already trained'); return json.load(open(run / 'metrics.json'))
    t0 = time.time(); tf.keras.backend.clear_session()
    m = vt.train(tf, {**V14, 'seed': seed}, part(S, 'train', drop), part(S, 'val', drop), ac.RICE_CLASSES, run)
    m.update({'seed': seed, 'dropped_sources': list(drop), 'minutes': round((time.time() - t0) / 60, 1),
              'lab_test': {'accuracy': lab_test_acc(run / 'rice_model.tflite', LAB_TEST), 'n': len(LAB_TEST)}})
    json.dump(m, open(run / 'metrics.json', 'w'), indent=1, default=float)
    print(f"{key}: val {m['val_accuracy']:.4f} | lab test {m['lab_test']['accuracy']:.4f} | {m['minutes']} min | {m['train_images']} train images"); return m
RICE_RUNS = {f'rice_v14_seed{s}': run_one(f'rice_v14_seed{s}', s) for s in SEEDS}
if CROSS_SOURCE_HOLDOUT:
    RICE_RUNS[f'rice_v14no{CROSS_SOURCE_HOLDOUT}_seed42'] = run_one(f'rice_v14no{CROSS_SOURCE_HOLDOUT}_seed42', 42, (CROSS_SOURCE_HOLDOUT,))

82420632/82420632 ━━━━━━━━━━━━━━━━━━━━ 3s 0us/step
Epoch 1/5
264/264 - 49s - 185ms/step - accuracy: 0.6308 - loss: 1.2890 - val_accuracy: 0.7389 - val_loss: 0.8526
Epoch 2/5
264/264 - 19s - 73ms/step - accuracy: 0.7398 - loss: 1.0571 - val_accuracy: 0.7629 - val_loss: 0.7520
Epoch 3/5
264/264 - 17s - 66ms/step - accuracy: 0.7602 - loss: 1.0112 - val_accuracy: 0.7785 - val_loss: 0.7125
Epoch 4/5
264/264 - 17s - 65ms/step - accuracy: 0.7636 - loss: 0.9837 - val_accuracy: 0.7893 - val_loss: 0.6644
Epoch 5/5
264/264 - 16s - 59ms/step - accuracy: 0.7778 - loss: 0.9721 - val_accuracy: 0.7707 - val_loss: 0.6679
Epoch 1/30
264/264 - 112s - 424ms/step - accuracy: 0.8436 - loss: 0.8097 - val_accuracy: 0.9130 - val_loss: 0.3019
Epoch 2/30
264/264 - 48s - 181ms/step - accuracy: 0.9120 - loss: 0.6585 - val_accuracy: 0.9430 - val_loss: 0.2299
Epoch 3/30
264/264 - 44s - 165ms/step - accuracy: 0.9261 - loss: 0.6269 - val_accuracy: 0.9334 - val_loss: 0.2883
Epoch 4/30
264/264 - 48s - 182ms/step - accur

/usr/local/lib/python3.13/dist-packages/tensorflow/lite/python/interpreter.py:457: UserWarning:     Warning: tf.lite.Interpreter is deprecated and is scheduled for deletion in
    TF 2.20. Please use the LiteRT interpreter from the ai_edge_litert package.
    See the [migration guide](https://ai.google.dev/edge/litert/migration)
    for details.
    
  warnings.warn(_INTERPRETER_DELETION_WARNING)


rice_v14_seed42: val 0.9790 | lab test 0.9925 | 28.7 min | 8442 train images
Epoch 1/5
264/264 - 43s - 162ms/step - accuracy: 0.6247 - loss: 1.2835 - val_accuracy: 0.7347 - val_loss: 0.8328
Epoch 2/5
264/264 - 19s - 71ms/step - accuracy: 0.7386 - loss: 1.0537 - val_accuracy: 0.7581 - val_loss: 0.7675
Epoch 3/5
264/264 - 19s - 71ms/step - accuracy: 0.7627 - loss: 1.0096 - val_accuracy: 0.7665 - val_loss: 0.7224
Epoch 4/5
264/264 - 19s - 71ms/step - accuracy: 0.7690 - loss: 0.9806 - val_accuracy: 0.7959 - val_loss: 0.6705
Epoch 5/5
264/264 - 16s - 60ms/step - accuracy: 0.7784 - loss: 0.9744 - val_accuracy: 0.7641 - val_loss: 0.6874
Epoch 1/30
264/264 - 109s - 413ms/step - accuracy: 0.8464 - loss: 0.8156 - val_accuracy: 0.9016 - val_loss: 0.3713
Epoch 2/30
264/264 - 49s - 184ms/step - accuracy: 0.9102 - loss: 0.6642 - val_accuracy: 0.9406 - val_loss: 0.2682
Epoch 3/30
264/264 - 48s - 183ms/step - accuracy: 0.9313 - loss: 0.6222 - val_accuracy: 0.9502 - val_loss: 0.2212
Epoch 4/30
264/264 

/usr/local/lib/python3.13/dist-packages/tensorflow/lite/python/interpreter.py:457: UserWarning:     Warning: tf.lite.Interpreter is deprecated and is scheduled for deletion in
    TF 2.20. Please use the LiteRT interpreter from the ai_edge_litert package.
    See the [migration guide](https://ai.google.dev/edge/litert/migration)
    for details.
    
  warnings.warn(_INTERPRETER_DELETION_WARNING)


rice_v14_seed7: val 0.9802 | lab test 0.9925 | 27.8 min | 8442 train images
Epoch 1/5
264/264 - 43s - 164ms/step - accuracy: 0.6162 - loss: 1.2959 - val_accuracy: 0.7215 - val_loss: 0.8487
Epoch 2/5
264/264 - 19s - 71ms/step - accuracy: 0.7373 - loss: 1.0577 - val_accuracy: 0.7677 - val_loss: 0.7416
Epoch 3/5
264/264 - 19s - 73ms/step - accuracy: 0.7610 - loss: 1.0055 - val_accuracy: 0.7839 - val_loss: 0.6811
Epoch 4/5
264/264 - 16s - 60ms/step - accuracy: 0.7716 - loss: 0.9842 - val_accuracy: 0.7743 - val_loss: 0.6941
Epoch 5/5
264/264 - 16s - 60ms/step - accuracy: 0.7769 - loss: 0.9644 - val_accuracy: 0.7839 - val_loss: 0.6654
Epoch 1/30
264/264 - 109s - 412ms/step - accuracy: 0.8449 - loss: 0.8068 - val_accuracy: 0.9022 - val_loss: 0.3258
Epoch 2/30
264/264 - 48s - 183ms/step - accuracy: 0.9088 - loss: 0.6685 - val_accuracy: 0.9268 - val_loss: 0.2583
Epoch 3/30
264/264 - 49s - 185ms/step - accuracy: 0.9321 - loss: 0.6145 - val_accuracy: 0.9460 - val_loss: 0.2270
Epoch 4/30
264/264 -

/usr/local/lib/python3.13/dist-packages/tensorflow/lite/python/interpreter.py:457: UserWarning:     Warning: tf.lite.Interpreter is deprecated and is scheduled for deletion in
    TF 2.20. Please use the LiteRT interpreter from the ai_edge_litert package.
    See the [migration guide](https://ai.google.dev/edge/litert/migration)
    for details.
    
  warnings.warn(_INTERPRETER_DELETION_WARNING)


rice_v14_seed2024: val 0.9790 | lab test 0.9882 | 22.7 min | 8442 train images
Epoch 1/5
246/246 - 42s - 173ms/step - accuracy: 0.6304 - loss: 1.2951 - val_accuracy: 0.7162 - val_loss: 0.8778
Epoch 2/5
246/246 - 18s - 72ms/step - accuracy: 0.7396 - loss: 1.0555 - val_accuracy: 0.7538 - val_loss: 0.7565
Epoch 3/5
246/246 - 17s - 67ms/step - accuracy: 0.7634 - loss: 1.0008 - val_accuracy: 0.7691 - val_loss: 0.7015
Epoch 4/5
246/246 - 18s - 73ms/step - accuracy: 0.7761 - loss: 0.9863 - val_accuracy: 0.7819 - val_loss: 0.6935
Epoch 5/5
246/246 - 15s - 60ms/step - accuracy: 0.7800 - loss: 0.9643 - val_accuracy: 0.7781 - val_loss: 0.6635
Epoch 1/30
246/246 - 108s - 440ms/step - accuracy: 0.8462 - loss: 0.8151 - val_accuracy: 0.8992 - val_loss: 0.3076
Epoch 2/30
246/246 - 45s - 184ms/step - accuracy: 0.9128 - loss: 0.6617 - val_accuracy: 0.9362 - val_loss: 0.2915
Epoch 3/30
246/246 - 45s - 185ms/step - accuracy: 0.9327 - loss: 0.6148 - val_accuracy: 0.9439 - val_loss: 0.2537
Epoch 4/30
246/24

/usr/local/lib/python3.13/dist-packages/tensorflow/lite/python/interpreter.py:457: UserWarning:     Warning: tf.lite.Interpreter is deprecated and is scheduled for deletion in
    TF 2.20. Please use the LiteRT interpreter from the ai_edge_litert package.
    See the [migration guide](https://ai.google.dev/edge/litert/migration)
    for details.
    
  warnings.warn(_INTERPRETER_DELETION_WARNING)


rice_v14noriceleafbd_seed42: val 0.9790 | lab test 0.9818 | 22.5 min | 7846 train images


In [11]:
# 7. Per-image records on the paper's held-out rice test photos (app decision code + deployed router), for the audit notebook
REC12 = OUT / 'agri_results_v12'; REC12.mkdir(exist_ok=True)
def run_set(set_name, items, interp, key):
    f = REC12 / f'rows_{set_name}__{key}.json'
    if f.exists(): return json.load(open(f))
    rows = []
    for i, (p, lab) in enumerate(items):
        with Image.open(p) as im:
            r = v12.replay_row(ac, im, lab, 'Rice', 'Rice', interp, ac.RICE_CLASSES, DEP['router'], None, tag=f'{set_name}__{key}:{i}')
        r['file'] = p; rows.append(r)
    json.dump(rows, open(f, 'w')); print(f'{set_name:18s} {key:26s} acc {100 * np.mean([r["pred"] == r["true"] for r in rows]):6.2f}% (n={len(rows)})')
    return rows
for set_name, (items, crop, _) in TESTS.items():
    for key in RICE_RUNS:  # noqa
        run_set(set_name, items, load_interp(MODELS / key / 'rice_model.tflite'), key)

rice_sethy         rice_v14_seed42            acc 100.00% (n=900)
rice_sethy         rice_v14_seed7             acc 100.00% (n=900)
rice_sethy         rice_v14_seed2024          acc 100.00% (n=900)
rice_sethy         rice_v14noriceleafbd_seed42 acc 100.00% (n=900)
rice_paddydoctor   rice_v14_seed42            acc  88.67% (n=600)
rice_paddydoctor   rice_v14_seed7             acc  87.50% (n=600)
rice_paddydoctor   rice_v14_seed2024          acc  86.00% (n=600)
rice_paddydoctor   rice_v14noriceleafbd_seed42 acc  86.50% (n=600)
rice_trichy        rice_v14_seed42            acc  96.76% (n=555)
rice_trichy        rice_v14_seed7             acc  99.10% (n=555)
rice_trichy        rice_v14_seed2024          acc  99.10% (n=555)
rice_trichy        rice_v14noriceleafbd_seed42 acc  98.38% (n=555)
rice_bd5           rice_v14_seed42            acc    nan% (n=0)


/usr/local/lib/python3.13/dist-packages/numpy/_core/fromnumeric.py:3904: RuntimeWarning: Mean of empty slice.
  return _methods._mean(a, axis=axis, dtype=dtype,
/usr/local/lib/python3.13/dist-packages/numpy/_core/_methods.py:147: RuntimeWarning: invalid value encountered in scalar divide
  ret = ret.dtype.type(ret / rcount)


rice_bd5           rice_v14_seed7             acc    nan% (n=0)
rice_bd5           rice_v14_seed2024          acc    nan% (n=0)
rice_bd5           rice_v14noriceleafbd_seed42 acc    nan% (n=0)
rice_dhan          rice_v14_seed42            acc  61.82% (n=55)
rice_dhan          rice_v14_seed7             acc  60.00% (n=55)
rice_dhan          rice_v14_seed2024          acc  63.64% (n=55)
rice_dhan          rice_v14noriceleafbd_seed42 acc  60.00% (n=55)
rice_riceleafbd    rice_v14_seed42            acc  98.65% (n=223)
rice_riceleafbd    rice_v14_seed7             acc  95.96% (n=223)
rice_riceleafbd    rice_v14_seed2024          acc  97.76% (n=223)
rice_riceleafbd    rice_v14noriceleafbd_seed42 acc  58.30% (n=223)


In [13]:
   TESTS = {k: v for k, v in TESTS.items() if v[0]}

In [14]:
# 8. Evaluation: new rice model vs the deployed and earlier models on the same photos (earlier numbers from the paper's records)
def old_rows(set_name, key):
    alias = {'rice_sethy': [('agri_results_followup', 'field_test')]}
    for d, s in alias.get(set_name, []) + [(n, set_name) for n in REC]:
        f = REC[d] / f'rows_{s}__{key}.json'
        if f.exists(): return json.load(open(f))
def acc(rows): return v12.rate(sum(r['pred'] == r['true'] for r in rows), len(rows)) if rows else None
def mcnemar(a, b):
    from scipy.stats import binomtest
    ra = {r['file']: r['pred'] == r['true'] for r in a}; rb = {r['file']: r['pred'] == r['true'] for r in b}
    common = [f for f in ra if f in rb]; x = sum(ra[f] and not rb[f] for f in common); y = sum(rb[f] and not ra[f] for f in common)
    return {'old_only_correct': x, 'new_only_correct': y, 'p_exact': float(binomtest(x, x + y, 0.5).pvalue) if x + y else 1.0}
summary = {'models': {}, 'per_class': {}, 'paired': {}}
for set_name in TESTS:
    m = {}
    for key in ['deployed', 'lab_seed42', 'adapted_seed42', 'multi_seed42']:
        r = old_rows(set_name, key)
        if r: m[f'old:{key}'] = acc(r)
    for key in RICE_RUNS:
        r = json.load(open(REC12 / f'rows_{set_name}__{key}.json')); m[f'v14:{key}'] = acc(r)
        summary['per_class'][f'{set_name}__{key}'] = v12.prf([x['true'] for x in r], [x['pred'] for x in r])
        for base in ('deployed', 'multi_seed42'):
            ob = old_rows(set_name, base)
            if ob and key.startswith('rice_v14_seed'): summary['paired'][f'{set_name}: {key} vs {base}'] = mcnemar([{**x, 'file': R(x['file'])} for x in ob], r)
    seeds_rows = [json.load(open(REC12 / f'rows_{set_name}__rice_v14_seed{s}.json')) for s in SEEDS]
    def vote(rows_list, tta):
        out = []
        for k in range(len(rows_list[0])):
            P = np.mean([np.mean([r[k]['probs']] + (r[k]['tta_probs'] if tta else []), axis=0) for r in rows_list], axis=0)
            out.append(ac.RICE_CLASSES[int(np.argmax(P))] == rows_list[0][k]['true'])
        return v12.rate(sum(out), len(out))
    m['v14:seed42+flipTTA'] = vote(seeds_rows[:1], True)
    if len(SEEDS) > 1: m[f'v14:ensemble{len(SEEDS)}+flipTTA'] = vote(seeds_rows, True)
    summary['models'][set_name] = m
summary['lab_test'] = {k: (v['lab_test'] or {}).get('accuracy') for k, v in RICE_RUNS.items()}
summary['notes'] = {'within_source': 'rice_sethy and rice_paddydoctor: photos from these surveys are in training (test photos and their near-copies excluded)',
                    'cross_source': f'only rice_v14no{CROSS_SOURCE_HOLDOUT}_seed42 on rice_{CROSS_SOURCE_HOLDOUT} is a cross-source number (that dataset was never trained on)' if CROSS_SOURCE_HOLDOUT else 'no cross-source model trained',
                    'app_model': 'the app uses one model without TTA (the seed with the best validation accuracy); ensemble+TTA rows are for reference',
                    'leaky': 'rice_trichy is known to overlap the rice laboratory collection; see the audit for its clean subset',
                    'no_farm_metadata': 'no farm/plot/date metadata exists; scene similarity is used as a proxy for farm separation',
                    'thresholds': THRESHOLD_SOURCE}
save('summary_v12.json', summary)
print('ACCURACY (%, 95% CI) on the paper test photos')
for s, m in summary['models'].items(): print(f'  {s:18s}', {k: (v['pct'], v['ci95']) for k, v in m.items()})
print('paired (exact McNemar):'); [print(f'  {k}: {v}') for k, v in summary['paired'].items()]
print('laboratory test split:', summary['lab_test'])

ACCURACY (%, 95% CI) on the paper test photos
  rice_sethy         {'old:deployed': (27.0, [24.2, 30.0]), 'old:lab_seed42': (27.0, [24.2, 30.0]), 'old:adapted_seed42': (97.33, [96.1, 98.2]), 'old:multi_seed42': (92.67, [90.8, 94.2]), 'v14:rice_v14_seed42': (100.0, [99.6, 100.0]), 'v14:rice_v14_seed7': (100.0, [99.6, 100.0]), 'v14:rice_v14_seed2024': (100.0, [99.6, 100.0]), 'v14:rice_v14noriceleafbd_seed42': (100.0, [99.6, 100.0]), 'v14:seed42+flipTTA': (100.0, [99.6, 100.0]), 'v14:ensemble3+flipTTA': (100.0, [99.6, 100.0])}
  rice_paddydoctor   {'old:deployed': (23.67, [20.4, 27.2]), 'old:lab_seed42': (25.83, [22.5, 29.5]), 'old:adapted_seed42': (36.0, [32.3, 39.9]), 'old:multi_seed42': (66.17, [62.3, 69.8]), 'v14:rice_v14_seed42': (88.67, [85.9, 91.0]), 'v14:rice_v14_seed7': (87.5, [84.6, 89.9]), 'v14:rice_v14_seed2024': (86.0, [83.0, 88.5]), 'v14:rice_v14noriceleafbd_seed42': (86.5, [83.5, 89.0]), 'v14:seed42+flipTTA': (89.0, [86.2, 91.3]), 'v14:ensemble3+flipTTA': (90.33, [87.7, 92.

In [15]:
# 9. Confusion matrices (figures saved to Drive)
import matplotlib; matplotlib.use('Agg'); import matplotlib.pyplot as plt
FIG = OUT / 'figures'; FIG.mkdir(exist_ok=True)
for key, p in summary['per_class'].items():
    labs = [l for l in p['labels'] if any(p['confusion'][l].values()) or any(p['confusion'][t][l] for t in p['labels'])]
    M = np.array([[p['confusion'][t][q] for q in labs] for t in labs])
    fig, ax = plt.subplots(figsize=(1 + 0.6 * len(labs), 1 + 0.6 * len(labs)))
    ax.imshow(M, cmap='Blues'); ax.set_xticks(range(len(labs))); ax.set_yticks(range(len(labs)))
    short = [l.replace('Rice_', '').replace('Maize_', '').replace('Disease_', '') for l in labs]
    ax.set_xticklabels(short, rotation=60, ha='right', fontsize=7); ax.set_yticklabels(short, fontsize=7)
    for i in range(len(labs)):
        for j in range(len(labs)): ax.text(j, i, M[i, j], ha='center', va='center', fontsize=6)
    ax.set_xlabel('predicted'); ax.set_ylabel('true'); ax.set_title(f"{key}  acc {p['accuracy']}%", fontsize=8)
    fig.tight_layout(); fig.savefig(FIG / f'confusion_{key}.png', dpi=130); plt.close(fig)

In [16]:
# 10. Save reproducibility metadata and the records archive for the audit notebook
import importlib.metadata as im_
pk = {}
for p in ('tensorflow', 'numpy', 'scipy', 'scikit-learn', 'torch', 'transformers', 'kagglehub', 'pillow'):
    try: pk[p] = im_.version(p)
    except Exception: pk[p] = None
meta = {'finished': time.strftime('%Y-%m-%d %H:%M:%S'), 'repo_commit': COMMIT, 'python': platform.python_version(), 'packages': pk, 'gpu': gpu_name,
        'seeds': SEEDS, 'train_args': V14, 'fractions': {'lab': LAB_FRACTIONS, 'field': FIELD_FRACTIONS},
        'thresholds': {'hash': HASH_THRESHOLD, 'source': THRESHOLD_SOURCE},
        'datasets': {h: str(r) for h, r in ROOTS.items()}, 'deployed_models': dep_info,
        'new_models': {p.parent.name: hashlib.sha256(p.read_bytes()).hexdigest() for p in MODELS.rglob('rice_model.tflite')}}
save('metadata_v12.json', meta)
for f in ('summary_v12.json', 'metadata_v12.json', 'leakage_split_report.json', 'data_report.json', 'inventory.json'):
    shutil.copy(OUT / f, REC12 / f)
for p in MODELS.glob('*/metrics.json'): shutil.copy(p, REC12 / f'metrics_{p.parent.name}.json')
FINAL = MODELS / 'final'; FINAL.mkdir(exist_ok=True)
BEST = max((k for k in RICE_RUNS if k.startswith('rice_v14_seed')), key=lambda k: RICE_RUNS[k]['val_accuracy'])
print('model for the app (best validation accuracy):', BEST)
for src, dst in ((BEST, 'rice_model_v14'),):
    d = MODELS / src
    shutil.copy(d / 'rice_model.tflite', FINAL / f'{dst}.tflite'); shutil.copy(d / 'labels.txt', FINAL / f'{dst}_labels.txt')
    if (d / 'rice_model_v14.keras').exists(): shutil.copy(d / 'rice_model_v14.keras', FINAL / 'rice_model_v14.keras')
print('final models for the app:', sorted(p.name for p in FINAL.iterdir()))
zp = OUT / 'agri_results_v12.zip'
with zipfile.ZipFile(zp, 'w', zipfile.ZIP_DEFLATED) as z:
    for f in sorted(REC12.glob('*.json')): z.write(f, Path('agri_results_v12') / f.name)
shutil.copy(zp, BASE / 'agri_audit_inputs' / 'agri_results_v12.zip') if (BASE / 'agri_audit_inputs').exists() else None
print('records for the audit:', zp, '(also copied to My Drive/agri_audit_inputs/)')
print('models:', sorted(str(p.relative_to(OUT)) for p in MODELS.rglob('*') if p.suffix in ('.tflite', '.h5', '.keras', '.json')))

model for the app (best validation accuracy): rice_v14_seed7
final models for the app: ['rice_model_v14.keras', 'rice_model_v14.tflite', 'rice_model_v14_labels.txt']
records for the audit: /content/drive/MyDrive/agri_v14/agri_results_v12.zip (also copied to My Drive/agri_audit_inputs/)
models: ['models/final/rice_model_v14.keras', 'models/final/rice_model_v14.tflite', 'models/rice_v14_seed2024/best.weights.h5', 'models/rice_v14_seed2024/metrics.json', 'models/rice_v14_seed2024/rice_model.tflite', 'models/rice_v14_seed2024/rice_model_v14.keras', 'models/rice_v14_seed42/best.weights.h5', 'models/rice_v14_seed42/metrics.json', 'models/rice_v14_seed42/rice_model.tflite', 'models/rice_v14_seed42/rice_model_v14.keras', 'models/rice_v14_seed7/best.weights.h5', 'models/rice_v14_seed7/metrics.json', 'models/rice_v14_seed7/rice_model.tflite', 'models/rice_v14_seed7/rice_model_v14.keras', 'models/rice_v14noriceleafbd_seed42/best.weights.h5', 'models/rice_v14noriceleafbd_seed42/metrics.json', 'mod

In [17]:
# Rotation/flip copy check: hash all 8 rotations/flips of each test photo vs every training photo
from PIL import ImageOps
S = SPLITS['rice']; train_paths = [S['items'][i][0] for i in S['split']['train'] + S['split']['val']]
Htr = hash_parallel(train_paths)
def dihedral_hashes(p):
    with Image.open(p) as im:
        im = ImageOps.exif_transpose(im).convert('L'); out = []
        for flip in (False, True):
            b = ImageOps.mirror(im) if flip else im
            for k in range(4):
                x = np.asarray(b.rotate(90 * k, expand=True).resize((8, 8), Image.Resampling.BILINEAR), dtype=np.float32)
                out.append(np.packbits(x.flatten() > x.mean()))
        return np.stack(out)
summary_rot = {}
for name, (items, _, _) in TESTS.items():
    flags = {p: bool(al.min_hamming(dihedral_hashes(p), Htr)[0].min() <= 4) for p, _ in items}
    summary_rot[name] = sum(flags.values())
    for key in RICE_RUNS:
        f = REC12 / f'rows_{name}__{key}.json'
        if not f.exists(): continue
        rows = json.load(open(f)); clean = [r for r in rows if not flags.get(r['file'], False)]
        a = 100 * np.mean([r['pred'] == r['true'] for r in rows]); c = 100 * np.mean([r['pred'] == r['true'] for r in clean]) if clean else float('nan')
        print(f"{name:18s} {key:28s} all {a:6.2f}% (n={len(rows)})  clean {c:6.2f}% (n={len(clean)})  rotated/flipped copies in training: {len(rows) - len(clean)}")
json.dump(summary_rot, open(OUT / 'rotation_copy_check.json', 'w')); print(summary_rot)

rice_sethy         rice_v14_seed42              all 100.00% (n=900)  clean 100.00% (n=47)  rotated/flipped copies in training: 853
rice_sethy         rice_v14_seed7               all 100.00% (n=900)  clean 100.00% (n=47)  rotated/flipped copies in training: 853
rice_sethy         rice_v14_seed2024            all 100.00% (n=900)  clean 100.00% (n=47)  rotated/flipped copies in training: 853
rice_sethy         rice_v14noriceleafbd_seed42  all 100.00% (n=900)  clean 100.00% (n=47)  rotated/flipped copies in training: 853
rice_paddydoctor   rice_v14_seed42              all  88.67% (n=600)  clean  89.63% (n=511)  rotated/flipped copies in training: 89
rice_paddydoctor   rice_v14_seed7               all  87.50% (n=600)  clean  88.26% (n=511)  rotated/flipped copies in training: 89
rice_paddydoctor   rice_v14_seed2024            all  86.00% (n=600)  clean  87.08% (n=511)  rotated/flipped copies in training: 89
rice_paddydoctor   rice_v14noriceleafbd_seed42  all  86.50% (n=600)  clean  87.87% 

## 11. Use the new rice model in the app (after you have looked at the results and the audit)
1. In **My Drive/agri_v14/models/final/**, right-click `rice_model_v14.tflite` → Share → *Anyone with the link* → copy the link. The file ID is the part between `/d/` and `/view`.
2. In the repo's `app.py`, set `RICE_MODEL_ID = "<that id>"`. The label order is unchanged (`agri_core.RICE_CLASSES`).
3. Run `pytest -q` before committing. Nothing in this notebook changes the repository; the router stays as deployed.

In [18]:
# 12. Try it on your own rice photos (upload one or more)
from google.colab import files
up = files.upload()
RICE_FINAL = load_interp(MODELS / 'final' / 'rice_model_v14.tflite')
for fn in up:
    with Image.open(fn) as im: img = ImageOps.exif_transpose(im).convert('RGB')
    rp = ac.predict_image(img, DEP['router']).astype(float); route = ac.route_crop(rp / rp.sum())
    d = v12.diagnose(ac, img, RICE_FINAL, ac.RICE_CLASSES, 'Rice')
    print(f"\n{fn}\n  deployed router: {route['crop']} ({route['confidence']:.1f}%)" + (' -> user must confirm the crop' if route['needs_review'] else ''))
    print(f"  rice v14: {d['label']} ({d['conf']:.1f}%), quadrants agreeing {d['agreement']}/4" + (' -> NEEDS REVIEW: ' + '; '.join(d['review_reasons']) if d['needs_review'] else ''))

Saving 122.JPG to 122.JPG


/usr/local/lib/python3.13/dist-packages/tensorflow/lite/python/interpreter.py:457: UserWarning:     Warning: tf.lite.Interpreter is deprecated and is scheduled for deletion in
    TF 2.20. Please use the LiteRT interpreter from the ai_edge_litert package.
    See the [migration guide](https://ai.google.dev/edge/litert/migration)
    for details.
    
  warnings.warn(_INTERPRETER_DELETION_WARNING)



122.JPG
  deployed router: Maize (99.0%) -> user must confirm the crop
  rice v14: Rice_Leaf_Blast (69.2%), quadrants agreeing 1/4 -> NEEDS REVIEW: Only 1 of 4 quadrants agree with the whole-leaf reading.
